# nb04 — лонг-континуация после кластера: стоп, funding, бета

**Кандидат из nb03 §1 (каузальная часть):** лонг на подтверждённом конце
кластера положителен на всех трёх окнах (как «−fade»: без стопа, без funding).

**Три вопроса, каждый может убить кандидата:**
1. Переживает ли лонг стоп снизу? (без стопа торговать нельзя — хвост вниз открыт)
2. Что съедает funding? (на разогнанных перпах его платит лонг)
3. **Не бета ли это?** Контроль: та же монета, вход за 48ч ДО пампа, тот же
   горизонт. Если ride ≈ ctrl — событие ничего не добавляет, это дрейф альтов.

Протокол прежний: разглядываем DEV, VALID/TEST — один взгляд в конце.
Породу nb02 можно применять только к ~17% событий (пересечение +5% до конца
кластера) — поправка nb03. Здесь она в §4, отдельно и осторожно.

In [1]:
import sys; sys.path.insert(0, '.')
from _lab import *

X = pd.read_parquet('_out/pump_ride.parquet')
X['entry'] = pd.to_datetime(X['entry'], utc=True)
def window(t):
    if t < pd.Timestamp('2025-07-01', tz='UTC'): return 'TRAIN'
    if t < pd.Timestamp('2026-02-01', tz='UTC'): return 'VALID'
    return 'TEST'
X['win'] = X.entry.map(window)
print('events:', len(X), '| symbols:', X.sym.nunique())
print(X.groupby('win').size().to_string())
print('funding coverage:', round(X.fund240.notna().mean()*100,0), '%')

events: 49834 | symbols: 573
win
TEST     14271
TRAIN    18933
VALID    16630
funding coverage: 31.0 %


## 1. Лонг × стоп × горизонт (без funding)

Стоп снизу: 5/10/20% и без стопа. Если весь результат живёт только «без стопа» —
кандидат нетороговый (открытый хвост).

In [2]:
for hz in (60, 240, 720):
    print(f'=== mean pnl %, exit +{hz}м ===')
    rows = []
    for tag, lab in (('s05','stop 5%'), ('s10','stop 10%'), ('s20','stop 20%'), ('s00','no stop')):
        r = {'стоп': lab}
        for w in ('TRAIN','VALID','TEST'):
            p = X[X.win == w][f'ride{hz}_{tag}'].dropna()
            r[w] = round(p.mean()*100, 2)
        rows.append(r)
    print(pd.DataFrame(rows).set_index('стоп').to_string()); print()

=== mean pnl %, exit +60м ===
          TRAIN  VALID  TEST
стоп                        
stop 5%   -0.15   0.08 -0.36
stop 10%  -0.07   0.24 -0.26
stop 20%  -0.01   0.34 -0.21
no stop    0.01   0.38 -0.19

=== mean pnl %, exit +240м ===
          TRAIN  VALID  TEST
стоп                        
stop 5%   -0.13   0.19 -0.58
stop 10%   0.06   0.44 -0.59
stop 20%   0.19   0.67 -0.49
no stop    0.25   0.76 -0.46

=== mean pnl %, exit +720м ===
          TRAIN  VALID  TEST
стоп                        
stop 5%    0.19   0.30 -0.73
stop 10%   0.60   0.63 -0.72
stop 20%   0.93   1.10 -0.53
no stop    1.09   1.23 -0.41



## 2. Funding

Средний funding-платёж лонга за горизонт (уже со знаком минус = издержка) и
итог net. Покрытие данных неполное — считаем на подмножестве с данными.

In [3]:
for hz in (240, 720):
    s = X[X[f'fund{hz}'].notna()]
    print(f'+{hz}м (n={len(s)}): mean funding {s[f"fund{hz}"].mean()*100:+.3f}% | '
          f'ride(s10) gross {s[f"ride{hz}_s10"].mean()*100:+.2f}% -> '
          f'net {(s[f"ride{hz}_s10"] + s[f"fund{hz}"]).mean()*100:+.2f}%')
    for w in ('TRAIN','VALID','TEST'):
        sw = s[s.win == w]
        print(f'   {w}: fund {sw[f"fund{hz}"].mean()*100:+.3f}%  net(s10) '
              f'{(sw[f"ride{hz}_s10"] + sw[f"fund{hz}"]).mean()*100:+.2f}%')

+240м (n=15474): mean funding +0.104% | ride(s10) gross +0.33% -> net +0.43%
   TRAIN: fund +0.059%  net(s10) +0.34%
   VALID: fund +0.164%  net(s10) +0.99%
   TEST: fund +0.220%  net(s10) -0.15%
+720м (n=15474): mean funding +0.233% | ride(s10) gross +0.88% -> net +1.11%
   TRAIN: fund +0.160%  net(s10) +1.11%
   VALID: fund +0.305%  net(s10) +1.76%
   TEST: fund +0.468%  net(s10) -0.06%


## 3. Бета-контроль — решающий вопрос

ctrl = та же монета, вход за 48ч до триггера, без стопа. Сравниваем с ride
без стопа (одинаковая механика). diff = ride − ctrl по каждому событию.

In [4]:
for hz in (60, 240, 720):
    print(f'=== +{hz}м ===')
    for w in ('TRAIN','VALID','TEST'):
        s = X[(X.win == w)].dropna(subset=[f'ride{hz}_s00', f'ctrl{hz}'])
        d = s[f'ride{hz}_s00'] - s[f'ctrl{hz}']
        print(f'  {w}: ride {s[f"ride{hz}_s00"].mean()*100:+.2f}%  ctrl {s[f"ctrl{hz}"].mean()*100:+.2f}%  '
              f'diff {d.mean()*100:+.2f}% (медиана diff {d.median()*100:+.2f}%)')
    print()

=== +60м ===
  TRAIN: ride +0.01%  ctrl -0.02%  diff +0.03% (медиана diff -0.13%)
  VALID: ride +0.38%  ctrl -0.15%  diff +0.53% (медиана diff -0.14%)
  TEST: ride -0.19%  ctrl +0.00%  diff -0.19% (медиана diff -0.36%)

=== +240м ===
  TRAIN: ride +0.23%  ctrl +0.22%  diff +0.01% (медиана diff -0.22%)
  VALID: ride +0.75%  ctrl -0.12%  diff +0.87% (медиана diff -0.24%)
  TEST: ride -0.46%  ctrl +0.31%  diff -0.77% (медиана diff -1.03%)

=== +720м ===
  TRAIN: ride +1.03%  ctrl +0.55%  diff +0.48% (медиана diff -0.04%)
  VALID: ride +1.16%  ctrl -0.41%  diff +1.57% (медиана diff -0.67%)
  TEST: ride -0.40%  ctrl +1.00%  diff -1.40% (медиана diff -1.80%)



## 4. Порода — только на каузальном подмножестве (~17%)

События, где пересечение +5% случилось ДО входа (t05 <= cend_lag): здесь
предсказание nb02 законно. Мало данных — только взгляд, не вывод.

In [5]:
B = pd.read_parquet('_out/breed_preds.parquet')
B['entry'] = pd.to_datetime(B['entry'], utc=True)
S = X[(X.t05 > 0) & (X.t05 <= X.cend_lag)].merge(
        B[['sym','entry','pred']], on=['sym','entry'], how='inner')
print('каузальное подмножество:', len(S))
DEV = S[S.entry < pd.Timestamp('2025-07-01', tz='UTC')].copy()
if len(DEV) > 300:
    DEV['q'] = pd.qcut(DEV.pred, 3, labels=['low','mid','high'])
    t = DEV.groupby('q', observed=True).agg(n=('pred','size'),
        r240=('ride240_s10', lambda s: round(s.mean()*100,2)),
        r720=('ride720_s10', lambda s: round(s.mean()*100,2)))
    print('DEV, терцили P(monster), лонг со стопом 10%:')
    print(t.to_string())

каузальное подмножество: 8341
DEV, терцили P(monster), лонг со стопом 10%:
        n  r240  r720
q                    
low   640 -0.59 -0.66
mid   639 -0.68 -1.06
high  640 -0.16  0.58


## 5. Каузальный признак на самом входе: сколько пампа удержано

runup_at_entry = цена входа / цена первого входа − 1 (известно на баре входа).
Гипотеза: удержанный рост (мало отдал) = сила; глубокий откат = слабость.
Только DEV.

In [6]:
DEV = X[X.entry < pd.Timestamp('2025-07-01', tz='UTC')].copy()
DEV = DEV.dropna(subset=['ride240_s10'])
DEV['q'] = pd.qcut(DEV.runup_at_entry, 5, labels=False, duplicates='drop')
t = DEV.groupby('q').agg(n=('runup_at_entry','size'),
    ru_med=('runup_at_entry', lambda s: round(s.median()*100,1)),
    r240=('ride240_s10', lambda s: round(s.mean()*100,2)),
    r720=('ride720_s10', lambda s: round(s.mean()*100,2)),
    ctrl240=('ctrl240', lambda s: round(s.mean()*100,2)))
print('DEV, квинтили runup_at_entry:')
print(t.to_string())

DEV, квинтили runup_at_entry:
      n  ru_med  r240  r720  ctrl240
q                                   
0  3787    -3.2 -0.10  0.53     0.53
1  3786    -1.4  0.67  1.33     0.24
2  3787    -0.3  0.09  0.50     0.25
3  3786     1.0 -0.23  0.27     0.08
4  3787     3.2 -0.13  0.35     0.01


## Выводы nb04

**🟢 Кандидат «лонг-континуация после кластера» убит, тремя способами.**

1. **Бета-контроль (решающий):** медианный diff (ride − ctrl той же монеты за
   48ч до пампа) отрицателен на ВСЕХ окнах и горизонтах; по средним TRAIN ≈ 0,
   VALID +0.9pp, TEST **−0.8…−1.4pp** — на TEST событие делает лонг ХУЖЕ
   обычного дрейфа монеты. Событийного edge нет; VALID-блеск — режим.
2. **Стоп-таблица:** TEST отрицателен при любом стопе и горизонте.
3. **Разгадка nb03:** его плюс был selection-lookahead (сэмпл «дошедших до +5%»,
   см. поправку 2 в nb03), а не edge; гипотеза о филе опровергнута замером (+0.02pp).

**Побочные факты.** Funding для лонга оказался попутным ветром (+0.10…+0.47%
за горизонт; ставки на этих перпах в среднем отрицательны) — но покрытие данных
31%, подмножество смещено. Порода на законных 17% (терциль high r720 +0.58 vs
low −0.66, DEV) — намёк в правильную сторону, n мал. runup_at_entry — немонотонен.

**Состояние линии после nb00–04:** на минутных кластерных событиях закрыты ОБЕ
стороны одиночного входа: шорт-фейд (nb01–03, три способа) и лонг-континуация
(nb04, контроль+стоп+разбор nb03). Класс «резкий памп на 1м OHLCV, один вход» —
исчерпан. Живые направления: (а) воспроизведение сильнейшего честного артефакта
старой линии — scale-in шорт + классификатор (в v3 ещё не тестировался!);
(б) грайнд-детектор — затяжные пампы, невидимые текущему детектору (наблюдение
пользователя); (в) секунды/микроструктура — среда обитания практиков.